# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [2]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [18]:
# TODO
df['revenue'] = df['qty'] * df['price']
total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()
print(f"The {len(df)} orders resulted in {total_units} total units sold, generating ${total_revenue:,.2f} in total revenue.")

df.head()

The 400 orders resulted in 783 total units sold, generating $8,520.00 in total revenue.


,vendor_id,category,qty,price,revenue
0,V-10,Drink,2,24.0,48.0
1,V-18,RainGear,1,12.0,12.0
2,V-18,Drink,3,4.5,13.5
3,V-10,Food,2,12.0,24.0
4,V-18,Drink,3,7.5,22.5


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [20]:
# TODO
by_category = df.groupby('category')[['revenue']].sum()
by_category['share (%)'] = (by_category['revenue'] / by_category['revenue'].sum()) * 100
by_category = by_category.sort_values('revenue', ascending=False)
top_category = by_category.index[0]
top_share = by_category['share (%)'].iloc[0]
print(f"The highest-earning category was {top_category}, which accounted for {top_share:.1f}% of the total revenue.")
by_category

The highest-earning category was Food, which accounted for 50.4% of the total revenue.


,revenue,share (%)
category,,
Food,4293.0,50.387324
Merch,1771.5,20.792254
Drink,1554.0,18.239437
RainGear,901.5,10.580986


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [21]:
# TODO
by_vendor = df.groupby('vendor_id').agg(
    avg_order_revenue=('revenue', 'mean'),
    order_count=('revenue', 'size')
)
by_vendor = by_vendor.sort_values('avg_order_revenue', ascending=False)
top_vendor = by_vendor.index[0]
top_avg = by_vendor['avg_order_revenue'].iloc[0]
top_count = by_vendor['order_count'].iloc[0]
print(f"Vendor {top_vendor} has the highest average order revenue at ${top_avg:.2f}, based on {top_count} orders. ")
by_vendor

Vendor V-01 has the highest average order revenue at $22.60, based on 94 orders. 


,avg_order_revenue,order_count
vendor_id,,
V-01,22.595745,94
V-18,21.750000,108
V-05,20.580645,93
V-10,20.314286,105


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [22]:
# TODO
merch_share = by_category.loc['Merch', 'share (%)']

print(f"Sale from Merch account for {merch_share:.1f}% of the total revenue.")

Sale from Merch account for 20.8% of the total revenue.


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [23]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor
joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')
assert len(joined) == len(df), "Row count changed during the merge!"
assert joined['revenue'].sum() == df['revenue'].sum(), "Revenue total changed during the merge!"
unmatched_vendor = joined.loc[joined['vendor_name'].isna(), 'vendor_id'].unique()[0]
unmatched_count = joined['vendor_name'].isna().sum()
joined['vendor_name'] = joined['vendor_name'].fillna('Unknown Vendor')
print(f"The merge preserved all {len(joined)} rows and revenue. Vendor {unmatched_vendor} lacked a name for {unmatched_count} orders, which were filled with 'Unknown Vendor'.")

joined.head()

The merge preserved all 400 rows and revenue. Vendor V-18 lacked a name for 108 orders, which were filled with 'Unknown Vendor'.


,vendor_id,category,qty,price,revenue,vendor_name
0,V-10,Drink,2,24.0,48.0,Cav Merch North
1,V-18,RainGear,1,12.0,12.0,Unknown Vendor
2,V-18,Drink,3,4.5,13.5,Unknown Vendor
3,V-10,Food,2,12.0,24.0,Cav Merch North
4,V-18,Drink,3,7.5,22.5,Unknown Vendor


**The unmatched vendor, and what I did about it: Vendor V-18 wasn't in the lookup table. I filled the missing values with 'Unknown Vendor' so their sales wouldn't just disappear when making the pivot table in Q6.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [25]:
# TODO
pivot_report = pd.pivot_table(
    joined,
    values='revenue',
    index='vendor_name',
    columns='category',
    aggfunc=np.sum,
    margins=True
)
grand_total = pivot_report.loc['All', 'All']
print(f"The pivot table report breaks down the ${grand_total:,.2f} in total revenue across all vendors and categories.")
pivot_report

The pivot table report breaks down the $8,520.00 in total revenue across all vendors and categories.


/tmp/ipykernel_4658/2896558977.py:2: FutureWarning: The provided callable <function sum at 0x79d47c1d2b60> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  pivot_report = pd.pivot_table(
/tmp/ipykernel_4658/2896558977.py:2: FutureWarning: The provided callable <function sum at 0x79d47c1d2b60> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  pivot_report = pd.pivot_table(
/tmp/ipykernel_4658/2896558977.py:2: FutureWarning: The provided callable <function sum at 0x79d47c1d2b60> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  pivot_report = pd.pivot_table(


category,Drink,Food,Merch,RainGear,All
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unknown Vendor,582.0,1018.5,508.5,240.0,2349.0
All,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [26]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a). FOr the next game, vendors should focus way more space on Food and Drink. Food is by far the biggest seller (making up about 50% of total revenue), while RainGear barely moves the needle (around 10%). Unless it's definitely going to rain, they are wasting stall space on umbrellas that could be used for food.

b). The least trustworthy part of this report is the unmatched vendor (V-18). Since this ID was missing from the lookup table, about 25% of all the orders belong to a vendor we can't even identify. Because of this missing data, we can't really audit their performance or give them any targeted feedback.